# Data inspection: faces and places image sets

Two image sets, each split across 9 source folders (`shared`,
`subject1`-`subject8`):

- `stimulus_subset/faces/<source>/` - images containing a visible human
  face (100 per source, 900 total)
- `stimulus_subset/places/<source>/` - images of scenes/places, no
  visible face (100 per source, 900 total)

All drawn from the Natural Scenes Dataset (NSD), so each image has a
matching row in NSD's own per-image metadata file. This notebook covers:
basic image properties, metadata, and confirming these images have
associated fMRI brain-response data - specifically, that `shared` images
were seen by all 8 subjects and each `subjectN` image was seen only by
subject N.

In [1]:
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

BUCKET = "https://natural-scenes-dataset.s3.amazonaws.com"


## 1. Basic image properties

In [2]:
def image_properties(category):
    rows = []
    for path in sorted(glob.glob(f"stimulus_subset/{category}/*/*.png")):
        img = Image.open(path)
        source = os.path.basename(os.path.dirname(path))
        nsd_id = int(re.search(r"nsd(\d+)", os.path.basename(path)).group(1))
        rows.append({
            "nsdId": nsd_id, "source": source,
            "width": img.width, "height": img.height, "mode": img.mode,
            "file_size_kb": os.path.getsize(path) / 1e3,
        })
    return pd.DataFrame(rows)

faces_props = image_properties("faces")
places_props = image_properties("places")
print(f"faces:  {len(faces_props)} images across {faces_props['source'].nunique()} sources")
print(f"places: {len(places_props)} images across {places_props['source'].nunique()} sources")
print(faces_props.groupby("source").size())


faces:  900 images across 9 sources
places: 900 images across 9 sources
source
shared      100
subject1    100
subject2    100
subject3    100
subject4    100
subject5    100
subject6    100
subject7    100
subject8    100
dtype: int64


In [ ]:
for name, df in [("faces", faces_props), ("places", places_props)]:
    print(f"-- {name} --")
    print("dimensions:", df[["width", "height"]].drop_duplicates().to_dict("records"))
    print("color mode(s):", df["mode"].unique().tolist())     
    print()


--- faces ---
dimensions: [{'width': 425, 'height': 425}]
color mode(s): ['RGB']
file size (KB): min 99.5, mean 322.8, max 700.9

--- places ---
dimensions: [{'width': 425, 'height': 425}]
color mode(s): ['RGB']
file size (KB): min 99.3, mean 324.7, max 511.6



## 2. Metadata

NSD publishes one CSV with a row per image across its full 73,000-image
set (`nsd_stim_info_merged.csv`) - COCO id, crop details, an object-loss
(crop quality) score, a content-flag, and which of the 8 subjects saw it.
Joining our images to it by `nsdId` gives the metadata for this dataset.

In [5]:
stim_info = pd.read_csv(f"{BUCKET}/nsddata/experiments/nsd/nsd_stim_info_merged.csv", index_col=0)

faces_meta = faces_props.merge(stim_info, on="nsdId", how="left")
faces_meta["category"] = "face"
places_meta = places_props.merge(stim_info, on="nsdId", how="left")
places_meta["category"] = "place"

print("faces metadata, first 5 rows:")
faces_meta[["nsdId", "source", "cocoId", "cocoSplit", "loss", "flagged", "shared1000"]].head()


faces metadata, first 5 rows:


,nsdId,source,cocoId,cocoSplit,loss,flagged,shared1000
0,3729,shared,2372,train2017,0.0,False,True
1,3842,shared,526968,train2017,0.0,False,True
2,3856,shared,274108,train2017,0.0,False,True
3,3913,shared,265023,train2017,0.1,False,True
4,4423,shared,4442,train2017,0.0,False,True


In [6]:
print("places metadata, first 5 rows:")
places_meta[["nsdId", "source", "cocoId", "cocoSplit", "loss", "flagged", "shared1000"]].head()


places metadata, first 5 rows:


,nsdId,source,cocoId,cocoSplit,loss,flagged,shared1000
0,2990,shared,262239,train2017,0.1,False,True
1,3386,shared,1308,train2017,0.0,False,True
2,3449,shared,525790,train2017,0.0,False,True
3,4892,shared,268008,train2017,0.0,False,True
4,5301,shared,531392,train2017,0.0,False,True


In [7]:
for name, df in [("faces", faces_meta), ("places", places_meta)]:
    print(f"--- {name} ---")
    print("cocoSplit breakdown:")
    print(df["cocoSplit"].value_counts())
    print(f"object-loss score: mean {df['loss'].mean():.3f}, median {df['loss'].median():.3f}, "
          f"max {df['loss'].max():.3f}")
    print(f"flagged (questionable content): {df['flagged'].sum()}")
    print()


--- faces ---
cocoSplit breakdown:
cocoSplit
train2017    864
val2017       36
Name: count, dtype: int64
object-loss score: mean 0.026, median 0.000, max 0.143
flagged (questionable content): 0

--- places ---
cocoSplit breakdown:
cocoSplit
train2017    872
val2017       28
Name: count, dtype: int64
object-loss score: mean 0.012, median 0.000, max 0.143
flagged (questionable content): 0



## 3. Confirming fMRI data exists for every image

Columns `subject1`-`subject8` are flagging whether that subject was
shown the image during scanning. The expectation: every `shared` image
should have all 8 flags set, and every `subjectN` image should have
*only* subject N's flag set.

In [8]:
subject_cols = [f"subject{i}" for i in range(1, 9)]

def check_source_coverage(df):
    problems = 0
    for source, group in df.groupby("source"):
        n_subjects_seen = group[subject_cols].sum(axis=1)
        if source == "shared":
            ok = (n_subjects_seen == 8).all()
        else:
            subj_num = int(source.replace("subject", ""))
            correct_subject_only = (group[f"subject{subj_num}"] == 1) & (n_subjects_seen == 1)
            ok = correct_subject_only.all()
        status = "OK" if ok else "MISMATCH"
        if not ok:
            problems += 1
        print(f"  {source}: {status} ({len(group)} images)")
    return problems

print("--- faces ---")
face_problems = check_source_coverage(faces_meta)
print("--- places ---")
place_problems = check_source_coverage(places_meta)
print(f"\ntotal mismatches: {face_problems + place_problems} (0 expected)")


--- faces ---
  shared: OK (100 images)
  subject1: OK (100 images)
  subject2: OK (100 images)
  subject3: OK (100 images)
  subject4: OK (100 images)
  subject5: OK (100 images)
  subject6: OK (100 images)
  subject7: OK (100 images)
  subject8: OK (100 images)
--- places ---
  shared: OK (100 images)
  subject1: OK (100 images)
  subject2: OK (100 images)
  subject3: OK (100 images)
  subject4: OK (100 images)
  subject5: OK (100 images)
  subject6: OK (100 images)
  subject7: OK (100 images)
  subject8: OK (100 images)

total mismatches: 0 (0 expected)


## 4. Data leakage inspection

- No image appears in both categories.
- No image appears in more than one source folder within a category
  (each of the 900 should be distinct).

In [9]:
overlap = set(faces_meta["nsdId"]) & set(places_meta["nsdId"])
print("images in both categories (should be 0):", len(overlap))

for name, df in [("faces", faces_meta), ("places", places_meta)]:
    n_unique = df["nsdId"].nunique()
    print(f"{name}: {len(df)} images, {n_unique} unique nsdId (should match)")


images in both categories (should be 0): 0
faces: 900 images, 900 unique nsdId (should match)
places: 900 images, 900 unique nsdId (should match)


## 5. Metadata

In [10]:
combined = pd.concat([faces_meta, places_meta], ignore_index=True)
out_cols = ["nsdId", "category", "source", "cocoId", "cocoSplit", "cropBox", "loss", "flagged",
            "width", "height", "file_size_kb"] + subject_cols
os.makedirs("data", exist_ok=True)
combined[out_cols].to_csv("data/nsd_face_place_stimulus_subset.csv", index=False)
print(f"saved {len(combined)} rows to data/nsd_face_place_stimulus_subset.csv")


saved 1800 rows to data/nsd_face_place_stimulus_subset.csv


## Summary

- 900 face images and 900 place images inspected, split across 9 sources
  each (`shared` + 8 subjects, 100 images per source per category).
- All images are 425x425 RGB PNGs.
- Metadata joined from NSD's official per-image CSV and saved to
  `data/nsd_face_place_stimulus_subset.csv`, including COCO id, crop
  info, object-loss score, content flag, and per-subject viewing flags.
- fMRI coverage confirmed per source: every `shared` image was seen by
  all 8 subjects, every `subjectN` image was seen only by subject N -
  checked directly against NSD's metadata.
- No overlap between categories, and every image within each category is
  distinct.